# Susceptibility and infectiousness

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/15-susceptibility-infectiousness-matrices.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Rows of a mixing matrix scale who gets infected; columns scale who infects.
This chapter shows those equivalences on a two-group density-dependent SIR,
and maps them onto summer4's preferred APIs.

**API asymmetry (honest):** infectiousness has a first-class hook —
{class}`~summer4.epi.ForceOfInfection` ``infectiousness`` / ``normalize_infectiousness`` on the force of
infection. There is no symmetric `add_susceptibility_adjustments`. Susceptibility
is expressed as a flow `adjust=` with {class}`~summer4.Multiply` and a
`where=` selector on the infection edges. Prefer that (or an equivalent
row-scaled matrix with `normalize="none"`) over baking susceptibility into
contact rates long-term — see `futureplans/foi-susceptibility-surface.md`.



## Shared setup

A closed two-group SIR with a unit (all-ones) mixing matrix and density-
dependent infection. Population shares live in `y0`.



In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    Multiply,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
    FlowModel,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("susceptible", "infectious", "recovered"))
group = Property("group", ("group1", "group2"))
pmap = PropertyMap.from_property(state).stratify(group)

END_TIME = 40.0
RECOVERY = 1.0 / 4.0
PROP1 = 0.4
SEED = 0.01
SCALER = 2.0
K_UNIT = np.ones((2, 2))

PLAN = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=np.linspace(0.0, END_TIME, int(END_TIME * 5) + 1),
)
params = {"risk": 0.5}


def y0_split() -> np.ndarray:
    y0 = np.zeros(pmap.size)
    sus = 1.0 - SEED
    y0[pmap.select(state["susceptible"] & group["group1"])] = PROP1 * sus
    y0[pmap.select(state["susceptible"] & group["group2"])] = (1.0 - PROP1) * sus
    y0[pmap.select(state["infectious"] & group["group1"])] = PROP1 * SEED
    y0[pmap.select(state["infectious"] & group["group2"])] = (1.0 - PROP1) * SEED
    return y0


def prevalence(res) -> pd.Series:
    i = np.asarray(res["comp"].select(state["infectious"]).values.data)
    return pd.Series(i.sum(axis=1), index=np.asarray(res["comp"].times.values))


def run_model(model: FlowModel):
    return model.compile().run(
        params, y0_split(), t0=0.0, t1=END_TIME, dt=0.1, save=PLAN, solver="euler"
    )


# Unstratified-style baseline: stratified but uniform susceptibility/infectiousness
base = FlowModel(pmap)

mixing = MixingMatrix(group, K_UNIT, normalize="none", check_reciprocal=False)
base.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=group,
    kind="density",
    contact_rate=Param("risk"),
    mixing=mixing,
),
    )
)
base.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))
res_base = run_model(base)
base_prev = prevalence(res_base)


## Increased susceptibility

Three equivalent constructions for making `group1` twice as susceptible:

1. `Multiply(scaler, where=group["group1"])` on the infection flow (preferred
   summer4 style — no first-class susceptibility FOI API).
2. Scale the **row** of the mixing matrix that infects `group1`.
3. Same as (1) with an explicit unit mixing matrix already attached.

Use `normalize="none"` when encoding susceptibility in the matrix; row
normalisation would cancel a uniform row scale.



In [ ]:
def infection_foi(matrix: np.ndarray) -> ForceOfInfection:
    return ForceOfInfection(
        "infection",
        infectious=state["infectious"],
        group_by=group,
        kind="density",
        contact_rate=Param("risk"),
        mixing=MixingMatrix(
            group, matrix, normalize="none", check_reciprocal=False
        ),
    )


# (1) Flow adjust only (unit mixing)
suscept_adjust = FlowModel(pmap)

mixing = MixingMatrix(group, K_UNIT, normalize="none", check_reciprocal=False)
suscept_adjust.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        infection_foi(K_UNIT),
        adjust=[Multiply(SCALER, where=group["group1"])],
    )
)
suscept_adjust.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))

# (2) Row-scaled matrix, no flow adjust
K_row = np.array([[SCALER, SCALER], [1.0, 1.0]])
suscept_matrix = FlowModel(pmap)

mixing = MixingMatrix(group, K_row, normalize="none", check_reciprocal=False)
suscept_matrix.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=group,
    kind="density",
    contact_rate=Param("risk"),
    mixing=mixing,
),
    )
)
suscept_matrix.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))

# (3) Unit matrix + flow adjust (same as 1; kept for the textbook contrast)
suscept_both = FlowModel(pmap)

mixing = MixingMatrix(group, K_UNIT, normalize="none", check_reciprocal=False)
suscept_both.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        infection_foi(K_UNIT),
        adjust=[Multiply(SCALER, where=group["group1"])],
    )
)
suscept_both.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))

s_adj = prevalence(run_model(suscept_adjust))
s_mat = prevalence(run_model(suscept_matrix))
s_both = prevalence(run_model(suscept_both))
np.testing.assert_allclose(s_adj.values, s_mat.values, atol=1e-6)
np.testing.assert_allclose(s_adj.values, s_both.values, atol=1e-6)
assert float(s_adj.max()) > float(base_prev.max())

suscept_out = pd.DataFrame(
    {
        "flow Multiply (group1)": s_adj,
        "row-scaled matrix": s_mat,
        "unit matrix + Multiply": s_both,
        "baseline": base_prev,
    }
)
suscept_out.plot(
    title="Increased susceptibility for group1 (three equivalent builds)",
    labels={"index": "time", "value": "infectious people"},
)


## Increased infectiousness

The same idea for infectiousness: scale a **column** of the mixing matrix,
or use {class}`~summer4.epi.ForceOfInfection` ``infectiousness`` / ``normalize_infectiousness``
(with `normalize=None` so the comparison is not renormalised away).



In [ ]:
# Infectiousness via FOI weights
inf_adj = FlowModel(pmap)

mixing = MixingMatrix(group, K_UNIT, normalize="none", check_reciprocal=False)
inf_adj.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=group,
    kind="density",
    contact_rate=Param("risk"),
    mixing=mixing,
    infectiousness={group["group1"]: SCALER, group["group2"]: 1.0},
    normalize_infectiousness=None,
),
    )
)
inf_adj.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))

# Column-scaled matrix
K_col = np.array([[SCALER, 1.0], [SCALER, 1.0]])
inf_matrix = FlowModel(pmap)

mixing = MixingMatrix(group, K_col, normalize="none", check_reciprocal=False)
inf_matrix.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=group,
    kind="density",
    contact_rate=Param("risk"),
    mixing=mixing,
),
    )
)
inf_matrix.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))

# Unit matrix + infectiousness adjustments (same as first)
inf_both = FlowModel(pmap)

mixing = MixingMatrix(group, K_UNIT, normalize="none", check_reciprocal=False)
inf_both.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state["infectious"],
        ForceOfInfection(
            "infection",
            infectious=state["infectious"],
            group_by=group,
            kind="density",
            contact_rate=Param("risk"),
            mixing=mixing,
            infectiousness={group["group1"]: SCALER, group["group2"]: 1.0},
            normalize_infectiousness=None,
        ),
    )
)
inf_both.add_flow(TransitionFlow(
    "recovery", state["infectious"], state["recovered"], RECOVERY
))

i_adj = prevalence(run_model(inf_adj))
i_mat = prevalence(run_model(inf_matrix))
i_both = prevalence(run_model(inf_both))
np.testing.assert_allclose(i_adj.values, i_mat.values, atol=1e-6)
np.testing.assert_allclose(i_adj.values, i_both.values, atol=1e-6)
assert float(i_adj.max()) > float(base_prev.max())

# Susceptibility and infectiousness are different processes.
assert float(np.max(np.abs(s_adj.values - i_adj.values))) > 1e-3

inf_out = pd.DataFrame(
    {
        "ForceOfInfection infectiousness": i_adj,
        "column-scaled matrix": i_mat,
        "unit matrix + infectiousness": i_both,
        "baseline": base_prev,
    }
)
inf_out.plot(
    title="Increased infectiousness for group1 (three equivalent builds)",
    labels={"index": "time", "value": "infectious people"},
)


## Summary

| Effect | Matrix view | Preferred summer4 API |
|---|---|---|
| Higher susceptibility of group $i$ | Scale row $i$ | `TransitionFlow(..., adjust=[Multiply(..., where=...)])` |
| Higher infectiousness of group $j$ | Scale column $j$ | `ForceOfInfection` ``infectiousness`` |
| Who mixes with whom | Off-diagonal structure | `MixingMatrix` |

Keep the mixing matrix for contact structure; use the dedicated infectiousness
path (and flow `adjust=` for susceptibility until a FOI-owned surface exists)
so those three intuitions stay separate.

